# ST-01 — Siretisation, phase 1 : SIRET exact

Pour chaque etablissement portant un SIRET, on cherche l'etablissement SIRENE
correspondant et on score la paire obtenue.

A noter : la version precedente du projet n'appliquait jamais le scoring APE ni
date en phase 3 de siretisation, parce que la requete de chargement des EG ne
selectionnait ni `cdape_stru` ni `dtouvertstruct_stru`. Les deux scores valaient
donc toujours `None` et la ponderation retombait systematiquement sur le score
global. Ce point est corrige : la phase 3 s'appuie desormais sur le referentiel
externe, et l'enrichissement est explicite dans ST-04.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.siretisation import matching_direct_siret, COLS_ETAB_EXPORT
from src.excel_export import export_phase1_excel, LABELS
from src.display import afficher_tableau, afficher_synthese
from config.settings import (
    FINESS_EGE_CLEAN, SIRENE_ETAB_CLEAN, ST_PHASE1, ST_VALIDES_P1,
    RESULTS_ST_DIR, PROCESSED_DIR,
)

RESULTS_ST_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VALIDES = {"VALIDE_FORT", "VALIDE"}

In [2]:
df_ege = pd.read_parquet(FINESS_EGE_CLEAN)

COLS_ETAB = ["siret", "siren", "denomination_norm_etab", "enseigne1_norm_etab",
             "enseigne2_norm_etab", "enseigne3_norm_etab",
             "denomination_usuelle_norm_etab", "code_commune_norm_etab",
             "numero_voie_norm_etab", "libelle_voie_complet_etab",
             "dept_etab"] + COLS_ETAB_EXPORT
df_etab = pd.read_parquet(SIRENE_ETAB_CLEAN,
                          columns=list(dict.fromkeys(COLS_ETAB)))

print(f"EGE FINESS  : {len(df_ege):,}")
print(f"Etab SIRENE : {len(df_etab):,}")

EGE FINESS  : 104,649
Etab SIRENE : 16,796,675


In [3]:
df_resultats = matching_direct_siret(df_ege, df_etab)
print()
print(df_resultats["statut"].value_counts())

EGE traités      : 104,649
   sans SIRET    : 13,104
   SIRET inconnu : 12,242
   à scorer      : 79,303


Phase 1 siretisation:   0%|          | 0/79303 [00:00<?, ?it/s]


statut
VALIDE           35948
VALIDE_FORT      24600
SANS_SIRET       13104
SIRET_INCONNU    12242
DOUTEUX          10562
REJETE            8193
Name: count, dtype: int64


In [4]:
afficher_tableau(
    df_resultats[df_resultats["statut"].isin(VALIDES)],
    "Apercu valides phase 1", max_lignes=9,
    colonnes=["NB_EgeId", "TX_NumFinessEge", "siret_norm_ege", "TX_NomEgeLong",
              "denominationUniteLegale", "nom_etab_retenu",
              "score_nom", "score_adresse", "score_global", "statut"])

NB_EgeId,TX_NumFinessEge,siret_norm_ege,TX_NomEgeLong,denominationUniteLegale,nom_etab_retenu,score_nom,score_adresse,score_global,statut
1,010000024,26010004500012,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,CENTRE HOSPITALIER FLEYRIAT,CENTRE HOSPITALIER FLEYRIAT,73.410000,100.000000,89.370000,VALIDE_FORT
2,010000032,26010003700068,CENTRE HOSPITALIER BUGEY SUD,CENTRE HOSPITALIER BUGEY SUD,CENTRE HOSPITALIER BUGEY SUD,100.000000,100.000000,100.000000,VALIDE_FORT
5,010000065,26010028400017,CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER,CENTRE HOSPITALIER MONTPENSIER TREVOUX,CENTRE HOSPITALIER MONTPENSIER TREVOUX,82.420000,100.000000,92.970000,VALIDE_FORT
7,010000081,26010010200011,CENTRE HOSPITALIER DU PAYS DE GEX,CENTRE HOSPITALIER DU PAYS DE GEX,CENTRE HOSPITALIER PAYS GEX,100.000000,100.000000,100.000000,VALIDE_FORT
8,010000099,26010013600019,CENTRE HOSPITALIER DE MEXIMIEUX,CH DE MEXIMIEUX,CH MEXIMIEUX,59.600000,88.760000,77.090000,VALIDE
9,010000107,26010020100052,CENTRE HOSPITALIER DE PONT DE VAUX,CH DE PONT DE VAUX,CH PONT VAUX,65.490000,70.000000,68.190000,VALIDE
12,010000131,20003004700076,CH INTERCOMMUNAL AIN VAL DE SAONE - THOISSEY,CENTRE HOSPITALIER INTERCOMMUNAL AIN VAL DE SAONE,CENTRE HOSPITALIER,22.630000,100.000000,69.050000,VALIDE
15,010000198,26011019200051,CENTRE HOSPITALIER PUBLIC DE HAUTEVILLE - UNITE INTERDEPT,CTRE HOSPITALIER PUBLIC D HAUTEVILLE,UNITE INTERDEPARTEMENTALE,38.390000,70.000000,57.360000,VALIDE
19,010000495,77554456200108,CENTRE PSYCHOTHERAPIQUE DE L'AIN,ORSAC,CTRE PSYCHOTHERAPIQUE AIN ST GEORGES,66.570000,70.000000,68.630000,VALIDE


In [5]:
COLS_COMPLET = [
    "ID_Ege", "NB_EgeId", "NB_PmSmsseId", "TX_NumFinessEge", "TX_Siret",
    "TX_CategorieEntiteGeographiqueExercice", "TX_NomEgeLong", "TX_NomEgeCourt",
    "TX_EtatActif", "cdcommune_norm_ege", "adresse_complete_ege",
    "siret_etab", "siren", "denominationUniteLegale",
    "enseigne1Etablissement", "denominationUsuelleEtablissement",
    "nom_etab_retenu", "adresse_complete_etab", "codeCommuneEtablissement",
    "categorieJuridiqueUniteLegale", "activitePrincipaleEtablissement",
    "dateCreationEtablissement", "etablissementSiege",
    "score_nom", "score_adresse", "score_global",
]
COLS_INFO = [
    "ID_Ege", "NB_EgeId", "NB_PmSmsseId", "TX_NumFinessEge", "TX_Siret",
    "TX_NomEgeLong", "cdcommune_norm_ege", "adresse_complete_ege",
]

compteurs = export_phase1_excel(
    df_resultats, ST_PHASE1, COLS_COMPLET, COLS_INFO,
    statuts_score=["VALIDE_FORT", "VALIDE", "DOUTEUX", "REJETE"],
    statuts_info=["SANS_SIRET", "SIRET_INCONNU"])

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  "Synthese siretisation phase 1")
print(f"\nFichier : {ST_PHASE1}")

Statut,Nb,% du total
Valide_fort,"24,600",23.5%
Valide,"35,948",34.4%
Douteux,"10,562",10.1%
Rejeté,"8,193",7.8%
Sans_SIRET,"13,104",12.5%
SIRET_inconnu,"12,242",11.7%
TOTAL,"104,649",100.0%



Fichier : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/siretisation/siretisation_phase1.xlsx


In [6]:
valides = df_resultats[df_resultats["statut"].isin(VALIDES)][
    ["NB_EgeId", "siret_norm_ege", "statut"]].copy()
valides.to_parquet(ST_VALIDES_P1, index=False)
print(f"{len(valides):,} EGE valides -> {ST_VALIDES_P1}")

60,548 EGE valides -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/siretisation_valides_p1.parquet
